# 🔌 Day 01a: Adapter & Decorator Patterns

---

## 🎯 What You'll Master Today
- **Adapter**: make incompatible interfaces work together
- **Decorator**: add behavior dynamically without modifying existing code
- Real-world examples: payment systems, logging, caching

---

## 🎭 The Analogies

**Adapter** = Travel power adapter  
Your Indian plug doesn't fit a US socket. The adapter translates between them — it doesn't change either one.

**Decorator** = Adding toppings to a pizza  
Base pizza stays the same. Each topping wraps it with extra flavor (and cost). You can stack as many as you want.

---

```
╔════════════════════════════════════════════════════════════════╗
║                    ADAPTER PATTERN                            ║
╠════════════════════════════════════════════════════════════════╣
║                                                                ║
║   Client ──→ Target Interface ──→ Adapter ──→ Adaptee          ║
║                                                                ║
║   Your code expects  ┐   Adapter   ┌ Legacy/3rd-party code    ║
║   send_email()        │  translates │ dispatch_mail()          ║
║                       └─────────────┘                          ║
║                                                                ║
╚════════════════════════════════════════════════════════════════╝
```

---

In [ ]:
# ============================================
# 1. ADAPTER — Legacy Payment System
# ============================================
from abc import ABC, abstractmethod

# Target interface — what our app expects
class PaymentProcessor(ABC):
    @abstractmethod
    def pay(self, amount: float) -> str: pass


# Modern system — already compatible
class StripePayment(PaymentProcessor):
    def pay(self, amount: float) -> str:
        return f"✅ Stripe: Charged ${amount:.2f}"


# Legacy system — INCOMPATIBLE interface
class LegacyPayPal:
    """Old 3rd-party SDK with different method names."""
    def make_payment(self, payment_amount: float, currency: str = "USD") -> dict:
        return {"status": "success", "amount": payment_amount, "currency": currency}


# Adapter — bridges the gap
class PayPalAdapter(PaymentProcessor):
    def __init__(self, paypal: LegacyPayPal):
        self._paypal = paypal
    
    def pay(self, amount: float) -> str:
        result = self._paypal.make_payment(amount)
        return f"✅ PayPal: Charged ${result['amount']:.2f} ({result['currency']})"


# Client code — works with ANY PaymentProcessor
def checkout(processor: PaymentProcessor, amount: float):
    print(processor.pay(amount))


checkout(StripePayment(), 99.99)
checkout(PayPalAdapter(LegacyPayPal()), 49.99)  # legacy works!

In [ ]:
# ============================================
# 2. DECORATOR — Adding Behavior Dynamically
# ============================================
from abc import ABC, abstractmethod

# Component interface
class Notifier(ABC):
    @abstractmethod
    def send(self, message: str) -> str: pass


# Concrete component
class EmailNotifier(Notifier):
    def send(self, message: str) -> str:
        return f"📧 Email: {message}"


# Base decorator
class NotifierDecorator(Notifier):
    def __init__(self, wrapped: Notifier):
        self._wrapped = wrapped
    
    def send(self, message: str) -> str:
        return self._wrapped.send(message)


# Concrete decorators — each adds behavior
class SMSDecorator(NotifierDecorator):
    def send(self, message: str) -> str:
        base = super().send(message)
        return f"{base}\n📱 SMS: {message}"


class SlackDecorator(NotifierDecorator):
    def send(self, message: str) -> str:
        base = super().send(message)
        return f"{base}\n💬 Slack: {message}"


class LoggingDecorator(NotifierDecorator):
    def send(self, message: str) -> str:
        print(f"[LOG] Sending notification: {message}")
        return super().send(message)


# Stack decorators!
notifier = EmailNotifier()
notifier = SMSDecorator(notifier)       # + SMS
notifier = SlackDecorator(notifier)     # + Slack
notifier = LoggingDecorator(notifier)   # + Logging

result = notifier.send("Server is down!")
print(result)

In [ ]:
# ============================================
# 3. Python Decorator vs GoF Decorator
# ============================================
import functools
import time

# Python's @decorator — syntactic sugar for function wrapping
def timer(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = func(*args, **kwargs)
        elapsed = time.perf_counter() - start
        print(f"⏱️ {func.__name__} took {elapsed:.4f}s")
        return result
    return wrapper

def retry(max_attempts: int = 3):
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            for attempt in range(1, max_attempts + 1):
                try:
                    return func(*args, **kwargs)
                except Exception as e:
                    print(f"  Attempt {attempt} failed: {e}")
                    if attempt == max_attempts:
                        raise
        return wrapper
    return decorator


# Stacking Python decorators
@timer
@retry(max_attempts=3)
def fetch_data(url: str) -> str:
    import random
    if random.random() < 0.6:
        raise ConnectionError("Network timeout")
    return f"Data from {url}"

try:
    result = fetch_data("https://api.example.com")
    print(result)
except ConnectionError:
    print("All retries failed")

```
╔═══════════════════════════════════════════════════════════════╗
║                  PATTERN COMPARISON MAP                       ║
╠════════════════════╦══════════════════════════════════════════╣
║  Aspect             ║  Adapter vs Decorator                   ║
╠════════════════════╬══════════════════════════════════════════╣
║  Purpose            ║  Adapter: convert interface              ║
║                     ║  Decorator: add behavior                 ║
║  Wraps              ║  Adapter: incompatible object            ║
║                     ║  Decorator: same-interface object        ║
║  Stacking           ║  Adapter: usually one layer              ║
║                     ║  Decorator: multiple layers              ║
║  Interface          ║  Adapter: changes interface              ║
║                     ║  Decorator: preserves interface          ║
╚════════════════════╩══════════════════════════════════════════╝
```

---

## 📝 Interview Questions

| # | Question | Key Point |
|---|----------|------------|
| 1 | What is the Adapter pattern? | Converts interface of one class to what the client expects. Bridge between incompatible interfaces |
| 2 | Adapter vs Facade? | Adapter wraps ONE object and changes its interface. Facade wraps MANY objects and simplifies |
| 3 | What is the Decorator pattern? | Wraps an object to add behavior without modifying it. Same interface in and out |
| 4 | Python @decorator vs GoF Decorator? | @decorator wraps functions. GoF Decorator wraps objects. Same concept, different scope |
| 5 | When to use Decorator over inheritance? | When you need to combine behaviors dynamically. Inheritance creates a class explosion |

---

## ┌──────────────────────────────────────────────────────────┐
## │              📌 KEY TAKEAWAYS                            │
## ├──────────────────────────────────────────────────────────┤
## │                                                          │
## │  • Adapter = interface translator (incompatible → ok)    │
## │  • Decorator = behavior stacker (same interface)         │
## │  • Adapter wraps a foreign object                        │
## │  • Decorator wraps a same-type object                    │
## │  • Python @decorator is the function-level version       │
## │  • Both follow Open/Closed Principle                     │
## │                                                          │
## └──────────────────────────────────────────────────────────┘

---

## ⏭️ Next Up: **Facade & Proxy** — simplify complex systems and control access